Unfolding of real AmBe neutron spectra works from ~2.5-5 MeV, but isn't fitting well any lower than that.

We suspect that our window is losing many low L neutrons that would contribute to unfolding the lower E regions.

To fix this, we're trying background subtraction. We will create a L spectrum histogram for the AmBe datasets and a background dataset (time-limited to match the calibration datasets), each with the same bins. We will then subtract the background counts from the AmBe counts. This should produce a L spectrum that includes the lower L neutrons without much gamma interference, which should lead to an unfolded spectrum that better matches the ISO AmBe spectrum.

## Initialization

In [ ]:
# Importing needed code

import re
import json
from collections import defaultdict
from functools import reduce
from typing import (
    Callable,
    # TypeVar,
    # Any,
    Literal
)
from datetime import datetime, timezone, timedelta
from math import sqrt, log
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LightSource
from scipy.integrate import cumulative_trapezoid
import pandas as pd
import numpy as np
from pint import Quantity

from data_processing.paths import (
    get_report_root, get_exp_root, get_reactor_data_root)
from data_processing.dataframe_validation import (
    DetectorDataframeColumn,
    BinningDataframeColumn,
    NonReactorDataframeColumn,
    SliceFitDataframeColumn
)
from data_processing.experiment_data_keys import (
    ExperimentDataKey,
    ExperimentNeutronData
)
from data_processing.loading.dataframe_loading import load_parquet_psd, load_caen_csvs
from data_processing.loading.timetag_processing import (
    calculate_timetag_hours,
    calculate_event_time
)
# from data_processing.processing.bimodal_fitting import (
#     get_psd_energy_histogram,
#     scan_histogram_slices,
#     find_failed_slices,
#     BimodalBounds,
#     BimodalParams
# )
from data_processing.processing.slice_fitting import (
    get_psd_energy_histogram, scan_histogram_slices, find_failed_slices)
from data_processing.processing.calibration import Detector, recalibrate
from data_processing.processing.neutron_classification import classify
from data_processing.reporting.plotting import plot_scatter, plot_classification
from data_processing import helpers
from data_processing.processing.neutron_window_strategy.strategy_factory import \
    NeutronStrategyFactory
from data_processing.processing.neutron_window_strategy.abstract_strategy import \
    AbstractNeutronStrategy
from data_processing.types import (
    NasaGenerationSettings,
    NeutronDistributionGenerationSettings,
    NeutronWindowSettings,
    BasicCutSettings,
    WindowType,
    SliceFitStyle,
    BimodalBounds,
    BimodalParams
)
from data_processing.loading.window_loading import (
    load_side_borders, get_neutron_window_paths)
from data_processing.loading.spectrum_unfolding import load_neutron_response_matrix
from data_processing.helpers.get_midpoints_from_bins import get_midpoints_from_bins
from data_processing.processing.spectrum_unfolding import (
    NDHistogram,
    unfold_spectrum,
    _nan_divide,
    zero_to_nan
)

In [ ]:
def bin_non_neutron_data(df, time_bins, data_col, selected_cols):
    start_time = time_bins[0]
    df = get_time_cut(df, 'Time', time_bins)

    binned_df = df.groupby("Time Bin", as_index=False)[data_col] \
        .agg(['mean', 'std']) \
        .copy()
    binned_df.columns = selected_cols
    binned_df['Bin midpoint'] = binned_df.index.to_series() \
        .apply(lambda x: x.mid)
    binned_df = bin_midpoint_time_to_seconds(binned_df, start_time)

    return binned_df

In [ ]:
def bin_midpoint_time_to_seconds(df, start_time):
    bin_mid_col = df[BinningDataframeColumn.BIN_MIDPOINT.value]
    bin_time_col_name = BinningDataframeColumn.BIN_TIME.value
    zeroed_midpoint = pd.to_datetime(bin_mid_col) - start_time
    df[bin_time_col_name] = zeroed_midpoint.dt.total_seconds()
    return df

In [ ]:
def get_time_cut(df, time_tag_col, time_bins):
    timetag_cut = pd.cut(df[time_tag_col], bins=time_bins)
    df[BinningDataframeColumn.TIME_BIN.value] = timetag_cut
    return df

In [ ]:
# fns ask questions, then generate strategy using factory

CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]


def get_nasa_loading_settings(
    calib_key: CalibrationKey
) -> str:
    left_border_type = helpers.get_input_with_default(
        """\
Which left border calculation do you want to use?
1: original left border (0.1966 MeVee)
2: newer left border (~0.1866 MeVee)
3: CAEN lower limit (0.050 MeVee) (default)
Press Enter for default
""",
        3,
        int
    )
    border_key: NasaBorderKey = (
        ExperimentDataKey.NASA_BORDERS if left_border_type == 1 
        else ExperimentDataKey.NASA_BORDERS_RECALC
    )
    file_name_prefix = f"{calib_key.value}_{border_key.value}"
    return file_name_prefix


def get_n_distro_loading_settings(
    calib_key: CalibrationKey
) -> str:
    file_name_prefix = f"{calib_key.value}_{ExperimentDataKey.N_WINDOW_BORDERS.value}"
    return file_name_prefix


def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> NasaGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = helpers.get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = helpers.get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = helpers.get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee)
2: newer (~0.1866 MeVee)
3: detector lower limit (0.050 MeVee) (default)
or press Enter for default
""",
            3,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = (
                ExperimentDataKey.NASA_BORDERS 
                if existing_left_border_version_input == 1 
                else ExperimentDataKey.NASA_BORDERS_RECALC
            )
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = get_neutron_window_paths(
                file_name_prefix=file_name_prefix)
            left_border, _ = load_side_borders(
                side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        elif existing_left_border_version_input == 3:
            lower_energy_bound = 0.05
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = helpers.get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.050)
""",
            0.050,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def get_n_distro_generation_settings(
) -> NeutronDistributionGenerationSettings:
    sigma = helpers.get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (3)
""",
        3,
        float
    )
    settings = NeutronDistributionGenerationSettings(
        sigma=sigma
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: NeutronStrategyFactory,
    window_type: WindowType,
    loading: bool,
    settings: NeutronWindowSettings
) -> Callable[[], AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data


## Experiment ID Input

In [ ]:
# Last ID = ID-418
# experiment_ids = helpers.input_experiment_ids()
# experiment_ids = ["ID-256"]
real_ids = ["ID-383.6", "ID-383.7", "ID-383.8"]
background_id = "ID-383.1"
experiment_ids = [*real_ids, background_id]

In [ ]:
experiment_ids

In [ ]:
# calib_input = helpers.get_input_with_default(
#     "Do you want to use new calibration? [y/n, or press Enter for yes]",
#     "y",
#     str
# )
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = (
    ExperimentDataKey.NEW_CALIBRATION
    if is_new_calibration
    else ExperimentDataKey.CAEN_CALIBRATION
)

In [ ]:
# detector_code = helpers.get_input_required(
#     """\
# Which detector was used?
# 1: Original detector (detector 1)
# 2: New detector (detector 2)
# """,
#     [Detector.ZERO, Detector.ONE],
#     lambda x: Detector(int(x)-1)
# )
detector_code = Detector.ZERO

In [ ]:
default_fit_input = 2  # changed to peak finder mode, approved by Fatima 2024-07-18
# fit_input = helpers.get_input_with_default(
#     """\
# Which bimodal fit type do you want to use?
# 1: Bounds based
# 2: Peak finder based (default)
# Press Enter for default
# """,
#     default_fit_input,
#     int
# )
fit_input = 1

fit_styles: dict[int, SliceFitStyle] = {
    1: "bounds",
    2: "peak_finder"
}
fit_style = fit_styles.get(fit_input, fit_styles[default_fit_input])

In [ ]:
# kind of window (Nasa, N distribution)
# load or generate
# specific settings for each condition to make namedtuple
# - generator settings (i.e. sigma, etc.)
# - file path prefix for loading
# done = False
strategy_factory = NeutronStrategyFactory()

# while not done:
#     window_input = helpers.get_input_with_default(
#         """\
# Which neutron classification window do you want to use?
# 1: NASA window (default)
# 2: Neutron distribution window
# Press Enter for default
# """,
#         1,
#         int
#     )
#     load_window_input = helpers.get_input_with_default(
#         """\
# Do you want to load the borders from the standard border file?
# [y/n, or press Enter for no]
# """,
#         "n",
#         str
#     )
#     done = True
#     will_load = load_window_input == "y"

#     try:
#         if window_input == 1:
#             if will_load:
#                 settings = get_nasa_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", True, settings
#                 )
#             else:
#                 settings = get_nasa_generation_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "nasa", False, settings
#                 )
#                 pass
#         elif window_input == 2:
#             if will_load:
#                 settings = get_n_distro_loading_settings(calib_key=calib_key)
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", True, settings
#                 )
#             else:
#                 settings = get_n_distro_generation_settings()
#                 factory_fn = make_strategy_factory_fn(
#                     strategy_factory, "n_distro", False, settings
#                 )
#         else:
#             print("Invalid classification window type given, please try again")
#             done = False
#     except ValueError as err:
#         print("Problem found:")
#         print(err)
#         print("Please try again")
#         done = False

# settings = NasaGenerationSettings(
#     window_offset=0.2,
#     sigma=5,
#     # lower_energy_bound=0.05,
#     lower_energy_bound=0,
#     recalculate_lower_energy_bound=False
# )
# class NeutronDistributionGenerationSettings(NamedTuple):
#     sigma: float = 3
#     lower_energy_bound: float = 0.1966,
#     upper_energy_bound: float = 0.688,
#     recalculate_lower_energy_bound: bool = False,
#     fom_energy_range: tuple[float, float] = (0.10, 0.35)
settings = NeutronDistributionGenerationSettings(
    sigma=5,
    lower_energy_bound=0,
    upper_energy_bound=2
)
factory_fn = make_strategy_factory_fn(
    strategy_factory, "n_distro", False, settings
)

In [ ]:
# bin_length = helpers.get_input_with_default(
#     "Enter bin length (in seconds), or press Enter for default (300 s)",
#     300,
#     int
# )
# bin_string = f"{bin_length}S"

In [ ]:
# bins_min = helpers.get_input_with_default(
#     "Enter minimum light output (in MeVee), or press Enter for default (0 MeVee)",
#     0,
#     float
# )
# bins_max = helpers.get_input_with_default(
#     "Enter maximum light output (in MeVee), or press Enter for default (10 MeVee)",
#     10,
#     float
# )
# bins_width = helpers.get_input_with_default(
#     "Enter light output bin width (in MeVee), or press Enter for default (0.02 MeVee)",
#     0.02,
#     float
# )
bins_min = 0
bins_max = 5
bins_width = 0.02

L_bins = np.arange(bins_min, bins_max + bins_width, bins_width).tolist()

In [ ]:
analysis_timestamp = datetime.now().strftime("%Y-%m%b-%d-%H-%M-%S")
overall_settings = {
    'calibration_type': repr(calib_key),
    'fitting_style': fit_style,
    'window_settings': repr(settings),
    # 'bin_length': bin_length
}

In [ ]:
analysis_timestamp

## Data Loading and Initial Processing

### Neutron

In [ ]:
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}
experiment_neutron_data = make_strategy_for_experiments(experiment_neutron_data, factory_fn)

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    # exp_data[ExperimentDataKey.UNCLASSIFIED] = load_parquet_psd(exp_id)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = load_caen_csvs(exp_id, raw=True)

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = calculate_timetag_hours(unclassified_df)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data[ExperimentDataKey.UNCLASSIFIED]
    unclassified_df = recalibrate(unclassified_df, detector_code)
    exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

### Response Matrix

In [ ]:
R = load_neutron_response_matrix(
    # Path("response_matrix_geant_point"),
    # Path("response_matrix_geant_gaussian"),
    # Path("response_matrix_geant_Tbird_WithSig"),
    # Path("response_matrix_R4_mono"),
    Path("response_matrix_11MeV"),
    min_L=bins_min,
    max_L=bins_max,
    L_bin_widths=bins_width
)
R = zero_to_nan(R)

### ISO AmBe Energy Spectrum

In [ ]:
ambe_file = Path("unfolding_test") / "ambe_spectrum.csv"
ambe_df = pd.read_fwf(ambe_file)
# ambe_limited = ambe_df.query("si1 <= 6")
ambe_E = ambe_df["si1"]
ambe_magnitude = ambe_df["sp1"]

## Data Processing

### Neutron Window

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 5e-3
energy_bins = np.arange(start=bins_min, stop=bins_max + energy_width, step=energy_width)
overall_settings['scan_idx'] = f"({start_scan_idx}, {end_scan_idx})"
overall_settings['energy_width'] = energy_width

for exp_id, exp_data in experiment_neutron_data.items():
    psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
    Z, xe, ye = get_psd_energy_histogram(
        psd_report,
        calibrated_energy_column,
        energy_bins=energy_bins,
        # energy_width=energy_width
    )
    exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
    exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
    exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
    exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
# get fit dataframe (not needed if loading, but do anyway to keep process consistent)
stop_here = False

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

    if fit_style == "bounds":
        # Default
        default_bounds: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.25, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.38, 0.04, 4000)
        )

        bounds_a: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.04, 4000)
        )

        bounds_b: BimodalBounds = (
            BimodalParams(0.1, 0.01, 1, 0.34, 0.01, 0),
            BimodalParams(0.2, 0.1, Z.max(), 0.36, 0.03, 4000)
        )

        # Ranged Example
        bounds = [
            ((0, 60), bounds_a),
        ]
    else:
        default_bounds = None
        bounds = None

    df, df_err = scan_histogram_slices(
        Z,
        xe,
        ye,
        fit_style=fit_style,
        default_bounds=default_bounds,
        bounds=bounds,
        start_idx=start_scan_idx,
        end_idx=end_scan_idx
    )
    df, bad_slice_indexes = find_failed_slices(df, exp_id, nan_total_threshold=10)

    if bad_slice_indexes is not None:
        exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
        exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
        stop_here = True
    else:
        # exp_data['fom_results'] = df
        exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    helpers.stop()

In [ ]:
# get borders from strategy
for exp_id, exp_data in experiment_neutron_data.items():
    if ExperimentDataKey.FOM_RESULTS not in exp_data:
        print(f"No good fit data on Experiment {exp_id}")
        continue

    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]

    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()

    exp_data[ExperimentDataKey.BORDERS] = borders

### Background Subtraction

In [ ]:
bg_data = experiment_neutron_data[background_id]
bg_Z = bg_data[ExperimentDataKey.PSD_HISTOGRAM]
bg_xe = bg_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
bg_ye = bg_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]

for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    
    if xe.shape != bg_xe.shape or not all(np.isclose(xe, bg_xe)):
        raise ValueError("Mismatch in x bins")
    if ye.shape != bg_ye.shape or not all(np.isclose(ye, bg_ye)):
        raise ValueError("Mismatch in y bins")
    
    Z_without_bg = Z - bg_Z
    exp_data["psd_bg_subtracted"] = Z_without_bg

### Apply Window

In [ ]:
def compute_masked_histogram(Z, xe, ye, border, res=10):
    """Adapted from ChatGPT solution"""
    m, n = Z.shape

    xe_left = xe[:-1]
    xe_right = xe[1:]
    ye_left = ye[:-1]
    ye_right = ye[1:]
    x_bin_sizes = xe_right - xe_left
    y_bin_sizes = ye_right - ye_left

    x_sub = (np.arange(res) + 0.5) / res
    y_sub = (np.arange(res) + 0.5) / res

    sub_x, sub_y = np.meshgrid(x_sub, y_sub)
    sub_x = sub_x[None, None, :, :]
    sub_y = sub_y[None, None, :, :]

    x0 = xe_left[:, None, None, None]
    x1 = xe_right[:, None, None, None]
    y0 = ye_left[None, :, None, None]
    y1 = ye_right[None, :, None, None]

    xv = x0 + sub_x * (x1 - x0)
    yv = y0 + sub_y * (y1 - y0)
    xv = xv + np.zeros_like(yv)
    yv = yv + np.zeros_like(xv)

    xv_flat = xv.reshape(-1)
    yv_flat = yv.reshape(-1)
    bottom_vals = border.bottom(xv_flat)
    top_vals = border.top(xv_flat)

    if border.left is None:
        left_mask = np.full_like(xv_flat, True)
    else:
        left_mask = (xv_flat >= border.left)
    if border.right is None:
        right_mask = np.full(xv_flat.shape, np.True_)
    else:
        right_mask = (xv_flat <= border.right)
    
    mask = (
        left_mask &
        right_mask &
        (yv_flat >= bottom_vals) &
        (yv_flat <= top_vals)
    )
    frac_coverage = mask.reshape(m, n, res, res).mean(axis=(2, 3))
    masked_counts = np.rint(Z * frac_coverage)
    return masked_counts

In [ ]:
for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data["psd_bg_subtracted"]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    borders = exp_data[ExperimentDataKey.BORDERS]
    masked_Z = compute_masked_histogram(Z, xe, ye, borders)
    exp_data["psd_windowed"] = masked_Z

### L Spectra

In [ ]:
# TODO turn PSD histogram into L spectra
# 1. Merge every 4 histogram bins along L axis (turns L width of 0.005 to width of 0.02)
# 2. Merge L edges to match (select every 4th value, starting from idx 0)
# 3. Sum along PSD axis
for exp_id, exp_data in experiment_neutron_data.items():
    Z = exp_data["psd_windowed"]
    xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
    ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
    Z_merged = np.add.reduceat(
        Z, np.arange(0, Z.shape[0], 4), axis=0
    )
    Z_spectra = Z_merged.sum(axis=1, keepdims=True)
    xe_spectra = xe[::4].copy()
    if xe_spectra[-1] != xe[-1]:
        xe_spectra = np.append(xe_spectra, xe[-1])
    L_mids = get_midpoints_from_bins(xe_spectra)
    PSD_mids = get_midpoints_from_bins(ye)
    reduced_PSD_mids = np.array([PSD_mids.mean()])
    L_spectrum = NDHistogram(Z_spectra, [L_mids, reduced_PSD_mids])
    phd_data = {"L_spectrum": L_spectrum}
    exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_data

In [ ]:
# # make PSD histogram
# R_L_mids, R_E_mids = R.midpoints
# reduced_E_mids = np.array([R_E_mids.mean()])
# for exp_id, exp_data in experiment_neutron_data.items():
#     neutrons_only = exp_data[ExperimentDataKey.UNCLASSIFIED]
#     # max_seconds = exp_data["max_seconds"]
#     neutron_energies = neutrons_only[calibrated_energy_column.value]
#     energy_bins = np.arange(start=bins_min, stop=bins_max + bins_width, step=bins_width)
    
#     # TODO generate neutron PHD histogram
#     Z_n, *_ = np.histogram(neutron_energies, bins=energy_bins)
#     # exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = {
#     #     "neutron": {"standard": Z_n, "standard_edges": energy_bins}
#     # }
#     PHD = Z_n.reshape(-1, 1)
#     PHD_mids = get_midpoints_from_bins(energy_bins)
#     if PHD_mids.shape[0] != R_L_mids.shape[0] or not all(np.isclose(PHD_mids, R_L_mids)):
#         raise ValueError("Bin mismatch!")
#     L_spectrum = NDHistogram(PHD, [R_L_mids, reduced_E_mids])
#     phd_data = {"L_spectrum": L_spectrum}
#     exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION] = phd_data

In [ ]:
# # make time-matched background histogram for each dataset
# R_L_mids, R_E_mids = R.midpoints
# reduced_E_mids = np.array([R_E_mids.mean()])
# bg_data = experiment_neutron_data[background_id]
# bg_df = bg_data[ExperimentDataKey.UNCLASSIFIED]
# bg_duration = bg_df["TIMETAG"].max()

# for exp_id, exp_data in experiment_neutron_data.items():
#     exp_df = exp_data[ExperimentDataKey.UNCLASSIFIED]

#     # make new BG L spectrum for matching duration
#     # if experiment duration is less than bg duration, start BG at 5 minutes in
#     exp_duration = exp_df["TIMETAG"].max()
#     if exp_duration == bg_duration:
#         start_time = 0
#         end_time = bg_duration
#     else:
#         start_time = 5 * 60 * 1e12
#         end_time = start_time + exp_duration
#     bg_df_time_limited = bg_df[bg_df["TIMETAG"].between(start_time, end_time)]
    
#     bg_energies = bg_df_time_limited[calibrated_energy_column.value]
#     energy_bins = np.arange(start=bins_min, stop=bins_max + bins_width, step=bins_width)
#     Z_n, *_ = np.histogram(bg_energies, bins=energy_bins)
#     bg_PHD = Z_n.reshape(-1, 1)
#     PHD_mids = get_midpoints_from_bins(energy_bins)
#     if PHD_mids.shape[0] != R_L_mids.shape[0] or not all(np.isclose(PHD_mids, R_L_mids)):
#         raise ValueError("Bin mismatch!")
#     bg_L_spectrum = NDHistogram(bg_PHD, [R_L_mids, reduced_E_mids])

#     # store in PHD data dictionary
#     phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
#     phd_data["background"] = bg_L_spectrum

### Interpolate ISO AmBe

In [ ]:
E_mids = R.midpoints[1]
iso_E_interp = np.interp(E_mids, ambe_E, ambe_magnitude)

fig, ax = plt.subplots()
# ax_2 = ax.twinx()
ax.plot(ambe_E, ambe_magnitude, color="blue")
ax.plot(E_mids, iso_E_interp, color="red")

In [ ]:
# Convert AmBe E spectrum
# We have no data about what this is measured in
# Just gotta try different correction factors and see what happens
# correction_factor = 3000
correction_factor = 0.4
iso_E_corrected = iso_E_interp * correction_factor
iso_E_corrected.shape

### Refolding with NaN

In [ ]:
R_counts = R.counts
iso_E_nand = np.where(iso_E_corrected==0, np.nan, iso_E_corrected)
iso_L_nand = np.nansum(R_counts * iso_E_nand, axis=1)
print(iso_L_nand.min())

### Unfolding with BG subtraction

In [ ]:
for real_id in real_ids:
    exp_data = experiment_neutron_data[real_id]
    phd_data = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]
    N_without_bg = phd_data["L_spectrum"]

    phi, unfold_info = unfold_spectrum(
        R, N_without_bg,
        apply_nan_each_iter=True,
        max_iterations=500,
        stability_E_min=0,
        stability_E_max=5,
        full_info=True,
        progress_report_interval=50
    )

    phi = zero_to_nan(phi)
    exp_data["phi"] = phi
    exp_data["unfolding_info"] = unfold_info

In [ ]:
# TODO get mean phi for all real AmBe datasets
real_phis = []
for real_id in real_ids:
    exp_data = experiment_neutron_data[real_id]
    phi = exp_data["phi"]
    real_phis.append(phi)
real_counts = [phi.counts for phi in real_phis]
real_midpoints = [phi.midpoints for phi in real_phis]
first_midpoints = real_midpoints[0]
L_mids_match = all([
    midpoints[0] == first_midpoints[0]
    for midpoints in real_midpoints
])
E_mids_match = all([
    (midpoints[1] == first_midpoints[1]).all()
    for midpoints in real_midpoints
])
if not L_mids_match or not E_mids_match:
    raise ValueError("Midpoints don't match!")
mean_counts = np.nansum(np.stack(real_counts), axis=0) / 3
mean_phi_nand = NDHistogram(mean_counts, first_midpoints)
print(mean_counts.shape)
print(real_counts[0].shape)

In [ ]:
# TODO get mean phis for unfolding info list for all real AmBe datasets
real_frame_phis = []
for real_id in real_ids:
    exp_data = experiment_neutron_data[real_id]
    phis = exp_data["unfolding_info"]["phis"]
    real_frame_phis.append(phis)

real_frame_phis = list(zip(*real_frame_phis))
real_frame_counts = [
    [phi.counts for phi in frame_phis]
    for frame_phis in real_frame_phis
]
real_frame_midpoints = [
    [phi.midpoints for phi in frame_phis]
    for frame_phis in real_frame_phis
]
first_midpoints = real_frame_midpoints[0][0]
L_mids_match = all([
    all([
        midpoints[0] == first_midpoints[0]
        for midpoints in frame_midpoints
    ])
    for frame_midpoints in real_frame_midpoints
])
E_mids_match = all([
    all([
        (midpoints[1] == first_midpoints[1]).all()
        for midpoints in frame_midpoints
    ])
    for frame_midpoints in real_frame_midpoints
])
if not L_mids_match or not E_mids_match:
    raise ValueError("Midpoints don't match!")
mean_frame_counts = [
    np.nansum(np.stack(frame_counts), axis=0) / 3
    for frame_counts in real_frame_counts
]
mean_phis_nand = [
    NDHistogram(mean_counts, first_midpoints)
    for mean_counts in mean_frame_counts
]

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"
transparent = "#00000000"

### Plots

#### Refolding

In [ ]:
fig, ax = plt.subplots()

ax.plot(R_L_mids, iso_L_nand, label="ISO")
for exp_id, exp_data in experiment_neutron_data.items():
    if exp_id == background_id:
        continue
    L_spectrum_normed = exp_data[ExperimentDataKey.PULSE_HEIGHT_DISTRIBUTION]["L_spectrum"]
    L_mids = L_spectrum_normed.midpoints[0]
    L_counts = L_spectrum_normed.counts.reshape(-1)
    ax.plot(L_mids, L_counts, label=exp_id)

ax.text(
    0.01, 0.99, f"Corr. factor = {correction_factor}",
    transform=ax.transAxes,
    va="top"
)

# ax.set_ylim(0, 6000)
ax.set_xlim(0, 2)
ax.set_yscale("log")

ax.set_xlabel("Light output (MeVee)", fontsize=fontsize)
ax.set_ylabel("Normalized counts", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
ax.legend()

#### Unfolding (BG subtraction)

In [ ]:
fig, ax = plt.subplots()

ax.plot(E_mids, iso_E_interp * correction_factor, label="ISO")
# for exp_id, exp_data in experiment_neutron_data.items():
#     phi = exp_data["limit_phi"]
#     real_E_mids = phi.midpoints[1]
#     real_E_counts = phi.counts.reshape(-1)
#     ax.plot(real_E_mids, real_E_counts, label=exp_id)
mean_E_mids = mean_phi_nand.midpoints[1]
mean_E_counts = mean_phi_nand.counts.reshape(-1)
ax.plot(mean_E_mids, mean_E_counts, label="Mean")

ax.set_xlabel("Pulse energy (MeV)", fontsize=fontsize)
ax.set_ylabel("Intensity (a.u.)", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
ax.legend()

In [ ]:
fig, ax = plt.subplots()

for real_id in real_ids:
    exp_data = experiment_neutron_data[real_id]
    chi = exp_data["unfolding_info"]["chis"]
    ax.plot(chi, label=exp_id)
    # real_E_mids = phi.midpoints[1]
    # real_E_counts = phi.counts.reshape(-1)
    # ax.plot(real_E_mids, real_E_counts, label=exp_id)

# ax.set_yscale("log")

ax.set_xlabel("Iteration", fontsize=fontsize)
ax.set_ylabel("Stopping criteria value", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)
ax.legend()

In [ ]:
fig, axs = plt.subplots(2, 1, sharex=True, figsize=(6, 12))
ax_hi, ax_lo = axs

for real_id in real_ids:
    exp_data = experiment_neutron_data[real_id]
    delta_Is = exp_data["unfolding_info"]["delta_Is"]
    ax_hi.plot(delta_Is, label=exp_id)
    ax_lo.plot([np.abs(delta_I) for delta_I in delta_Is], label=exp_id)

ax_lo.set_xlim(0, 250)

ax_hi.set_yscale("symlog", linthresh=1e-9)
ax_lo.set_yscale("log")

ax_lo.set_xlabel("Iteration", fontsize=fontsize)
ax_hi.set_ylabel(r"$\Delta{I}$", fontsize=fontsize)
ax_lo.set_ylabel(r"$abs(\Delta{I})$", fontsize=fontsize)
ax_hi.tick_params(labelsize=fontsize)
ax_lo.tick_params(labelsize=fontsize)

ax_hi.legend()

In [ ]:
iterations = [24, 49, 74, 99, 149, 199, 249]
# iterations = [0, 1, 2, 3]
for iteration in iterations:
    chis = []
    delta_Is = []
    fig, ax = plt.subplots()
    
    ax.plot(E_mids, iso_E_interp * correction_factor, label="ISO")
    for real_id in real_ids:
        exp_data = experiment_neutron_data[real_id]
        # phi = exp_data["limit_unfolding_info"]["phis"][iteration]
        chi = exp_data["unfolding_info"]["chis"][iteration]
        delta_I = exp_data["unfolding_info"]["delta_Is"][iteration]
        chis.append(chi)
        delta_Is.append(delta_I)
        # real_E_mids = phi.midpoints[1]
        # real_E_counts = phi.counts.reshape(-1)
        # ax.plot(real_E_mids, real_E_counts, label=exp_id)
    mean_phi = mean_phis_nand[iteration]
    mean_E_mids = mean_phi.midpoints[1]
    mean_E_counts = mean_phi.counts.reshape(-1)
    ax.plot(mean_E_mids, mean_E_counts, label="Mean")
    
    ax.text(
        0.99, 0.99, f"Iter. {iteration+1}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=fontsize/2
    )
    chis_text = [f"{chi:.2e}" for chi in chis]
    chis_text = ",".join(chis_text)
    ax.text(
        0.99, 0.94, f"Chis = {chis_text}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=fontsize/2
    )
    delta_Is_text = [f"{delta_I:.2e}" for delta_I in delta_Is]
    delta_Is_text = ",".join(delta_Is_text)
    ax.text(
        0.99, 0.89, fr"$\Delta$I = {delta_Is_text}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=fontsize/2
    )
    
    ax.set_xlim(0, 5)
    
    ax.set_xlabel("Pulse energy (MeV)", fontsize=fontsize)
    ax.set_ylabel("Intensity (a.u.)", fontsize=fontsize)
    ax.tick_params(labelsize=fontsize)
    fig.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(E_mids, iso_E_interp * correction_factor, label="ISO", color="blue")
# phis = redo_unfolding_info["phis"]

cmap = plt.colormaps["viridis"]
norm = mpl.colors.Normalize(vmin=0, vmax=len(mean_phis_nand)-1)

for i, phi in enumerate(mean_phis_nand):
    color = cmap(norm(i))
    real_E_mids = phi.midpoints[1]
    real_E_counts = phi.counts.reshape(-1)
    ax.plot(real_E_mids, real_E_counts, alpha=0.2, color=color)

ax.set_xlabel("Pulse energy (MeV)", fontsize=fontsize)
ax.set_ylabel("Intensity (a.u.)", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)

In [ ]:
fig, ax = plt.subplots()
exp_id = "ID-383.6"
exp_data = experiment_neutron_data[exp_id]
L_spectrum = exp_data["L_spectrum_normed"]
test_spectrums = exp_data["limit_unfolding_info_nand"]["test_spectrums"]

ax.plot(L_spectrum.midpoints[0], L_spectrum.counts, label="Original", color="blue")

cmap = plt.colormaps["viridis"]
norm = mpl.colors.Normalize(vmin=0, vmax=len(test_spectrums)-1)

for i, test_spectrum in enumerate(test_spectrums):
    color = cmap(norm(i))
    test_spec_mids = test_spectrum.midpoints[0]
    test_spec_counts = test_spectrum.counts.reshape(-1)
    ax.plot(test_spec_mids, test_spec_counts, alpha=0.2, color=color)

ax.set_yscale("log")
ax.set_xlim(0, 0.1)
# ax.set_xlim(0, 2)
# ax.set_ylim(0, 500)

ax.set_xlabel("Pulse energy (MeVee)", fontsize=fontsize)
ax.set_ylabel("Intensity (a.u.)", fontsize=fontsize)
ax.tick_params(labelsize=fontsize)